In [3]:
from hydra import compose, initialize
from omegaconf import OmegaConf
import json
import os
from pathlib import Path
import glob
with initialize(config_path="../../../configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg.paths))

/tmp/ipykernel_3632/2997671192.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="../../../configs"):


checkpoints: ${paths.results}/checkpoints
totalsegmri: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalsegmri
totalseg: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalseg
nako_orig: /nfs/data/nii/data0/GNC/GNC_759
nako: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/nako
nnunet: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/results/nnunet
results: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/results/patch_icl



In [4]:
# get predifined test split
splits_path = '/nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/splits_966.json'
with open(splits_path, 'r') as f:
    splits = json.load(f)
subjects_test = splits['test']
print(len(subjects_test))

290


In [10]:
import nibabel as nib
import numpy as np
import shutil

img_base_path = Path(cfg.paths.nako_orig) / "links"
mask_base_path = Path(cfg.paths.nako_orig) / "data"
img_target = "30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii"
mask_target = "30/opportunistic-screening/seg.nii.gz"

### images have 4 channels :"0": "opp", "1": "in","2": "fat","3": "water"
### for now, only save "in" channel 

def process_subject(subject, img_base_path, mask_base_path, img_target, mask_target, images_dir, labels_dir):
    img_path = img_base_path / subject / img_target
    mask_path = mask_base_path / subject / mask_target
    
    # --- Images: split 4D -> 4x 3D channel files ---
    print(f"Looking for images in: {img_path}")
    files = glob.glob(str(img_path))
    print(f"Subject: {subject}, Files found: {len(files)}")

    if len(files) > 0:
        img = nib.load(files[0])
        data = img.get_fdata()  # (320, 260, 316, 4)

        if data.ndim == 4:
            for ch in range(data.shape[-1]):
                if ch == 1:  # Only save the "in" channel
                    channel_data = data[..., ch].astype(np.float32)
                    new_img = nib.Nifti1Image(channel_data, img.affine, img.header)
                    new_img.header.set_data_shape(channel_data.shape)
                    dst = os.path.join(images_dir, f"{subject}_{ch:04d}.nii.gz")
                    nib.save(new_img, dst)
                    print(f"  Saved channel {ch} -> {dst}")
        else:
            # skip if not 4D
            print(f"  Warning: Image is not 4D, skipping: {files[0]}")

    # --- Mask: enforce 3D header and save ---
    print(f"Looking for masks in: {mask_path}")
    mask_files = glob.glob(str(mask_path))
    print(f"Subject: {subject}, Mask files found: {len(mask_files)}")

    if len(mask_files) > 0:
        mask = nib.load(mask_files[0])
        mask_data = mask.get_fdata().astype(np.uint8)

        # Squeeze out any phantom 4th dim
        if mask_data.ndim == 4:
            mask_data = mask_data[..., 0]

        new_mask = nib.Nifti1Image(mask_data, mask.affine, mask.header)
        new_mask.header.set_data_shape(mask_data.shape)
        dst = os.path.join(labels_dir, f"{subject}.nii.gz")
        nib.save(new_mask, dst)
        print(f"  Saved mask -> {dst}")


from concurrent.futures import ProcessPoolExecutor, as_completed

def run_parallel(subjects, images_dir, labels_dir, max_workers=16):
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                process_subject,
                subject,
                img_base_path, mask_base_path,
                img_target, mask_target,
                images_dir, labels_dir,
            ): subject
            for subject in subjects
        }
        for future in as_completed(futures):
            subject = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"ERROR processing {subject}: {e}")


In [11]:
run_parallel(subjects_test[:3],
             os.path.join(cfg.paths.nako, "imagesTs"),
             os.path.join(cfg.paths.nako, "labelsTs"))

Looking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100963/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100161/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100263/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii


Subject: 100963, Files found: 1Subject: 100161, Files found: 1Subject: 100263, Files found: 1


  Saved channel 1 -> /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/nako/imagesTs/100161_0001.nii.gz
Looking for masks in: /nfs/data/nii/data0/GNC/GNC_759/data/100161/30/opportunistic-screening/seg.nii.gz
Subject: 100161, Mask files found: 1
  Saved channel 1 -> /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/nako/imagesTs/100963_0001.nii.gz  Saved channel 1 -> /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/nako/imagesTs/100263_0001.nii.gz

Look